Load Data

In [45]:
import pandas as pd
df = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")
df.head()


,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_usd_transaction_value,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity


In [46]:
print("Shape:", df.shape)
df.info()
print("Missing values:\n", df.isnull().sum())
print("Duplicated rows:\n", df.duplicated().sum())
print("Payment Systems:\n", df["system"].value_counts())
#Check duplicate months inside each system
print("Duplicated system-month records:\n", df.duplicated(subset=["system", "report_month"]).sum())

Shape: (214, 50)
<class 'pandas.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 50 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   report_month                     214 non-null    str    
 1   report_year                      214 non-null    int64  
 2   report_quarter                   214 non-null    str    
 3   system                           214 non-null    str    
 4   users_total                      214 non-null    int64  
 5   transactions_total               214 non-null    int64  
 6   transaction_value_jod            214 non-null    int64  
 7   user_jordanian_count             214 non-null    int64  
 8   user_non_jordanian_count         214 non-null    int64  
 9   user_male_count                  214 non-null    int64  
 10  user_female_count                214 non-null    int64  
 11  user_age_under_30_count          214 non-null    int64  
 12  user_age_31_50_c

Convert report_month to datetime

In [47]:
df["report_month"] = pd.to_datetime(df["report_month"], format="%Y-%m")
print(df["report_month"].dtype)

#Sort the dataframe by system and report_month
df = df.sort_values(["system", "report_month"]).reset_index(drop=True)
print(df[["report_month", "system"]].head())

#Extract the month number
df["month"] = (df["report_month"].dt.month)
df.head()

datetime64[us]
  report_month system
0   2023-01-01    ACH
1   2023-02-01    ACH
2   2023-03-01    ACH
3   2023-04-01    ACH
4   2023-05-01    ACH


,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,month
0,2023-01-01,2023,Q1,ACH,0,875000,706000000,0,0,0,...,253,2700000,34,190000,0,0,0.0,0.0,Low Activity,1
1,2023-02-01,2023,Q1,ACH,0,819000,639000000,0,0,0,...,294,3200000,39,390000,0,0,0.0,0.0,Low Activity,2
2,2023-03-01,2023,Q1,ACH,0,920000,736000000,0,0,0,...,267,2800000,34,360000,0,0,0.0,0.0,Normal Activity,3
3,2023-04-01,2023,Q2,ACH,0,893000,673000000,0,0,0,...,201,1600000,35,420000,0,0,0.0,0.0,Low Activity,4
4,2023-05-01,2023,Q2,ACH,0,931000,757000000,0,0,0,...,387,2900000,47,520000,0,0,0.0,0.0,Normal Activity,5


Encode report_quarter

In [48]:
quarter_mapping = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}
df["report_quarter"] = (df["report_quarter"].map(quarter_mapping))
print(df["report_quarter"].unique())
print("Missing encoded quarters:", df["report_quarter"].isnull().sum())

[1 2 3 4]
Missing encoded quarters: 0


Encode activity_level

In [49]:
activity_mapping = {
    "Low Activity": 0,
    "Normal Activity": 1,
    "High Activity": 2
}
df["activity_level"] = (df["activity_level"].map(activity_mapping))
print(df["activity_level"].unique())
print("Missing encoded activities:", df["activity_level"].isnull().sum())

[0 1 2]
Missing encoded activities: 0


Decode activity_level

In [50]:
activity_decoding = {
    0: "Low Activity",
    1: "Normal Activity",
    2: "High Activity"
}

Encode system

In [ ]:
df["system_name"] = df["system"]

df = pd.get_dummies(
    df,
    columns=["system"],
    prefix="system",
    dtype=int
)

system_columns = [
    column
    for column in df.columns
    if column.startswith("system_")
    and column != "system_name"
]

print(system_columns)

print(system_columns)

df[system_columns].head()

['system_ACH', 'system_CliQ', 'system_ECCU', 'system_JoMoPay', 'system_eFAWATEERcom']
['system_ACH', 'system_CliQ', 'system_ECCU', 'system_JoMoPay', 'system_eFAWATEERcom']


,system_ACH,system_CliQ,system_ECCU,system_JoMoPay,system_eFAWATEERcom
0,1,0,0,0,0
1,1,0,0,0,0
2,1,0,0,0,0
3,1,0,0,0,0
4,1,0,0,0,0


Decode system

Convert ACH foreign-currency values to JOD

In [52]:
USD_TO_JOD = 0.709
EUR_TO_JOD = 0.770
GBP_TO_JOD = 0.910

df["ach_usd_value_jod"] = (df["ach_usd_transaction_value"] * USD_TO_JOD)

df["ach_eur_value_jod"] = (df["ach_eur_transaction_value"] * EUR_TO_JOD)

df["ach_gbp_value_jod"] = (df["ach_gbp_transaction_value"] * GBP_TO_JOD)

In [53]:
display(df.head())
print("Final shape:", df.shape)
print(df.dtypes)
print(df.isnull().sum())

,report_month,report_year,report_quarter,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,user_female_count,...,month,system_name,system_ACH,system_CliQ,system_ECCU,system_JoMoPay,system_eFAWATEERcom,ach_usd_value_jod,ach_eur_value_jod,ach_gbp_value_jod
0,2023-01-01,2023,1,0,875000,706000000,0,0,0,0,...,1,ACH,1,0,0,0,0,22191700.0,2079000.0,172900.0
1,2023-02-01,2023,1,0,819000,639000000,0,0,0,0,...,2,ACH,1,0,0,0,0,19639300.0,2464000.0,354900.0
2,2023-03-01,2023,1,0,920000,736000000,0,0,0,0,...,3,ACH,1,0,0,0,0,23397000.0,2156000.0,327600.0
3,2023-04-01,2023,2,0,893000,673000000,0,0,0,0,...,4,ACH,1,0,0,0,0,19143000.0,1232000.0,382200.0
4,2023-05-01,2023,2,0,931000,757000000,0,0,0,0,...,5,ACH,1,0,0,0,0,26233000.0,2233000.0,473200.0


Final shape: (214, 59)
report_month                       datetime64[us]
report_year                                 int64
report_quarter                              int64
users_total                                 int64
transactions_total                          int64
transaction_value_jod                       int64
user_jordanian_count                        int64
user_non_jordanian_count                    int64
user_male_count                             int64
user_female_count                           int64
user_age_under_30_count                     int64
user_age_31_50_count                        int64
user_age_51_plus_count                      int64
customer_individual_count                   int64
customer_legal_entity_count                 int64
wallet_individual_count                     int64
wallet_commercial_count                     int64
user_existing_count                         int64
user_new_count                              int64
tx_purchases_count         

Feature selection

In [54]:
regression_features = [
    "month",                         # Monthly seasonality
    "report_year",                   # Changes in activity across years
    "transactions_total",            # Overall monthly transaction volume
    "users_total",                   # Total number of users
    "customer_legal_entity_count",   # CliQ legal-entity participation
    "wallet_commercial_count",       # JoMoPay commercial-wallet activity
    "tx_money_transfers_count",      # CliQ and JoMoPay transfer activity
    "ef_digital_payment_count",      # eFAWATEERcom digital-payment activity
    "ach_jod_transaction_count",     # ACH JOD transaction activity
    "returned_cheque_count"          # ECCU returned-cheque risk
] + system_columns
 
classification_features = regression_features.copy()  # same set used for both tasks
 
print("Regression features:", len(regression_features))
print("Classification features:", len(classification_features))

Regression features: 15
Classification features: 15


In [55]:
X = df[regression_features].copy()

y_reg = df["transaction_value_jod"].copy()

y_clf = df["activity_level"].copy()

print("X shape:", X.shape)
print("Regression target shape:", y_reg.shape)
print("Classification target shape:", y_clf.shape)

X shape: (214, 15)
Regression target shape: (214,)
Classification target shape: (214,)


In [56]:

from sklearn.model_selection import train_test_split


train_indices, test_indices = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

print("Training records:", len(train_indices))
print("Testing records:", len(test_indices))

Training records: 171
Testing records: 43


In [57]:
X_train_reg = X.loc[train_indices].copy()
X_test_reg = X.loc[test_indices].copy()

y_train_reg = y_reg.loc[train_indices].copy()
y_test_reg = y_reg.loc[test_indices].copy()

print("Regression Data:")

print("X_train_reg:", X_train_reg.shape)
print("X_test_reg:", X_test_reg.shape)
print("y_train_reg:", y_train_reg.shape)
print("y_test_reg:", y_test_reg.shape)

Regression Data:
X_train_reg: (171, 15)
X_test_reg: (43, 15)
y_train_reg: (171,)
y_test_reg: (43,)


In [58]:
X_train_clf = X.loc[train_indices].copy()
X_test_clf = X.loc[test_indices].copy()

y_train_clf = y_clf.loc[train_indices].copy()
y_test_clf = y_clf.loc[test_indices].copy()

print("Classification Data:")

print("X_train_clf:", X_train_clf.shape)
print("X_test_clf:", X_test_clf.shape)
print("y_train_clf:", y_train_clf.shape)
print("y_test_clf:", y_test_clf.shape)

Classification Data:
X_train_clf: (171, 15)
X_test_clf: (43, 15)
y_train_clf: (171,)
y_test_clf: (43,)


In [59]:
import os
import json
import joblib
 
 
joblib.dump(
    {
        "X_train": X_train_clf,
        "X_test": X_test_clf,
        "y_train": y_train_clf,
        "y_test": y_test_clf
    },
    f"preprocessed_classification.pkl"
)
 
joblib.dump(
    {
        "X_train": X_train_reg,
        "X_test": X_test_reg,
        "y_train": y_train_reg,
        "y_test": y_test_reg
    },
    f"preprocessed_regression.pkl"
)
 
# Save the decoding dictionaries too (regression.py / classification.py /
# the Flask app will all need them to show human-readable labels).
with open(f"decoding_dicts.json", "w") as f:
    json.dump(
        {
            "activity_decoding": activity_decoding,
            "activity_mapping": activity_mapping,
            "regression_features": regression_features,
            "classification_features": classification_features,
        },
        f,
        indent=2
    )
 
print("Files ready for regression.py and classification.py to load independently.")

Files ready for regression.py and classification.py to load independently.
